# Creazione training set

## Downloads e setup
Il notebook ha lo scopo di creare un dataset di vocalizzi di mammiferi marini con label. Il dataset prodotto sarà poi utilizziato per effettuare clustering sulle diverse specie e per il riconoscimento dell'eventuale presenza o meno di vocalizzi in un file audio (call detection). <br>
Per l'esecuzione del notebook bisogna avere a disposizione:
- il dataset completo dei vocalizzi divisi per specie di [Watkins](https://whoicf2.whoi.edu/science/B/whalesounds/fullCuts.cfm)
- un csv contenente i metadati dei file
- il dataset delle [registrazioni estese](https://whoicf2.whoi.edu/science/B/whalesounds/masterFiles.cfm) da cui sono stati ritagliati i vocalizzi 

In [20]:
import pandas as pd
import os
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from scipy import signal
from pathlib import Path
from scipy.io import wavfile
from ds_download import download_dataset, retrieve_metadata, check_full_audio, link_generator

DOWNLOAD_FILES = False
DOWLOAD_METADATA = False
CHECK_MASTER_TAPES = False

Vengono quindi scaricati tutti i file dei vocalizzi ritagliati organizzati in una cartella chiamata 'dataset' che conterrà le sottocartelle con i file divisi per specie.

In [21]:
url = 'https://whoicf2.whoi.edu/science/B/whalesounds/fullCuts.cfm'

ds_path = 'dataset'
md_path = 'metadata'
mt_path = 'master_tapes'

if not os.path.exists(ds_path):
    os.makedirs(ds_path)
if not os.path.exists(md_path):
    os.makedirs(md_path)
if not os.path.exists(mt_path):
    os.makedirs(mt_path)

if DOWNLOAD_FILES:
    download_dataset(ds_path, url, 'getSpecies', 'pickYear')

Scaricati tutti i file si effettua il conteggio di file presenti per specie salvato poi in un file chiamato 'species_cont.csv' nella cartella 'metadata'.

In [22]:
folder_list = []
for folder in os.listdir(ds_path):
    folder_path = Path(ds_path, folder)
    if os.path.isdir(folder_path):
        file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(Path(folder_path, f))])
        folder_list.append({'species': folder, 'file_count': file_count})

species_df = pd.DataFrame(folder_list)
species_df.to_csv(Path(md_path, 'species_count.csv'), index=False)

Per ogni file audio di ogni specie vengono scaricati i metadati di interesse e salvati in un file chiamato 'metadata.csv' salvato nella cartella 'metadata'. Il file contiene le seguenti colonne:
- filename - il nome del file
- CU: - un timestamp codificato per indicare punto di inizio e fine del vocalizzo all'interno della registrazione estesa
- SR: - il sample rate del file audio
- CS: - la durata del vocalizzo 
- species - la specie di appartenenza

In [23]:
base_url = 'https://whoicf2.whoi.edu/science/B/whalesounds/metaData.cfm?RN='

# Campi dei metadati da estrarre
col = ['filename', 'CU:', 'SR:', 'CS:']
metadata_df = pd.DataFrame(columns=col)

if DOWLOAD_METADATA:
    for folder in os.listdir('dataset'):
        folder_dict = retrieve_metadata(Path(ds_path, folder), base_url, col)
        metadata_df = pd.concat([metadata_df, pd.DataFrame.from_records(folder_dict)], ignore_index=True)
    metadata_df.to_csv(Path(md_path, 'metadata.csv'), index=False)

Per ogni file del dataset si verifica se esiste o meno il file della registrazione completa da cui è stato estratto il ritaglio. <br>I primi 5 caratteri del filename di ogni ritaglio rappresentano il nome della registrazione completa di appartenenza con il seguente formato:
- anno: due cifre
- mese: due cifre
- numero progressivo per anno: una cifra

Il nome della registrazione completa, specie presente ed un booleano (che indica l'esistenza effettiva del file) vengono salvati in un file chiamato 'full_audio.csv' nella cartella 'metadata'.

In [24]:
if CHECK_MASTER_TAPES:   
    md_df = pd.read_csv(Path(md_path, 'metadata.csv'))
    md_df['filename'] = md_df['filename'].apply(lambda x: x[:5])
    md_df = md_df.drop_duplicates(subset=['filename'])
    df_list = []

    df_list = md_df.apply(check_full_audio, args=(df_list,), axis=1)
    full_audio_df = pd.DataFrame(df_list, columns=['species', 'filename', 'full_audio'])
    full_audio_df.to_csv(Path(md_path,'full_audio.csv'), index=False)

Per scaricare i file delle registrazioni complete vengono considerate solo le specie che hanno più di 1000 registrazioni ritagliate. <br>
Viene generato un file di testo contenente tutti i link per i download necessari. <br>
Il download effettivo viene fatto tramite software esterno per velocizzare l'operazione. I file audio estesi vengono salvati in una cartella chiamata 'master_tapes'

In [25]:
base_url = "https://whoicf2.whoi.edu/science/B/whalesounds/WhaleSounds/MasterFiles/"
full_audio_df = pd.read_csv(Path(md_path, 'full_audio.csv'))
download_df = full_audio_df[full_audio_df['full_audio'] == True]


popular_species = pd.read_csv(Path(md_path, 'species_count.csv'))
popular_species = popular_species[popular_species['file_count'] > 999]

print(popular_species)

restricted_link = download_df[download_df['species'].isin(popular_species['species'])]
link_df = restricted_link.apply(link_generator, args=(base_url,), axis=1)
with open(Path(md_path,'link_list.txt'), 'w') as f:
    for link in link_df:
        for l in link:
            f.write(f"{l}\n")


                        species  file_count
27                 Killer Whale        2647
30      Long-Finned Pilot Whale        1213
36  Pantropical Spotted Dolphin        1034
44                  Sperm Whale        1422


Viene quindi filtrato il file originale dei metadati in modo che contenga solo i ritagli delle specie più numerose (più di 1000 files) e per cui esiste una registrazione estesa. Viene aggiunta anche una colonna per mantenere il nome della registrazione estesa di appartenenza. Viene salvato tutto in un nuovo file, 'filtered_metadata.csv' nella cartella 'metadata'.

In [26]:
md_df = pd.read_csv(Path(md_path, 'metadata.csv'))
md_df = md_df[md_df['species'].isin(popular_species['species'])]

md_df['master_tape'] = md_df['filename'].apply(lambda x: x[:5])
md_df['master_tape'] = md_df['master_tape'].astype('int64')

md_df = md_df[md_df['master_tape'].isin(restricted_link['filename'])]
md_df.to_csv(Path(md_path, 'filtered_metadata.csv'), index=False)

# Analisi metadati

In [36]:
import re

fmd_df = pd.read_csv(Path(md_path, 'filtered_metadata.csv')) 
unique_df_list = []

# Lista di dataframe uno per ogni 'master_tape'
mt_list = fmd_df['master_tape'].unique()
unique_mt = [fmd_df[fmd_df['master_tape'] == mt] for mt in mt_list]

# Funzione per identificare i diversi pattern della colonna 'CU:'
def extract_pattern(row):
    value = row['CU:']
    if pd.isnull(value):
        return "NULL"
    
    # Rimuozione spazi
    value = str(value).strip()
    pattern = re.sub(r'\d+', 'num', value)
    pattern = re.sub(r'\.', '.', pattern)
    return pattern

def hour_norm(value):
    if pd.isnull(value):
        return 0
    value = str(value).strip()
    parts = value.split(':')
    if len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    return value

# Se un master tape ha più pattern, viene rimosso dalla lista
for df in unique_mt:
    pattern = df.apply(extract_pattern, axis=1)
    unique_pattern = pattern.unique()
    if len(unique_pattern) == 1 and unique_pattern[0] != 'num':
        unique_df_list.append(df)
        print(df['master_tape'].iloc[0], unique_pattern)
        df['CS:'] = df['CS:'].apply(hour_norm)
        tot_len = df['CS:'].astype(float).sum()
        print(tot_len)

print(len(unique_df_list))


90033 ['num:num.num-num:num.num']
3229.5299999999997
60012 ['num  Bnum:num.num  num:num.num']
90.48400000000001
64030 ['num  Bnum:num  num:num.num']
52.11699999999999
64031 ['num  Bnum:num   num:num.num']
440.253
97761 ['num:num:num:num']
106.20499999999998
97764 ['num:num:num:num']
30.8307
97765 ['num:num:num:num']
13.1297
97768 ['num:num:num:num']
15.383000000000001
97772 ['num:num:num:num']
44.09092
97773 ['num:num:num:num']
30.25238
97775 ['num:num:num:num']
35.96746
97776 ['num:num:num:num']
40.8675
97777 ['num:num:num:num']
7.9719999999999995
97778 ['num:num:num:num']
24.642599999999998
97779 ['num:num:num:num']
37.0457
97780 ['num:num:num:num']
27.457400000000003
53007 ['num  Bnum:num.num  num:num.num']
59.368
54004 ['num  Bnum:num.num  num:num.num']
277.154
56034 ['num  Bnum:num.num  num:num.num']
41.137
56035 ['num  Bnum:num.num  num:num.num']
15.811
58001 ['num  Bnum:num  num:num.num']
83.899
75004 ['num  Bnum:num  num:num.num']
210.776
75012 ['num  Bnum:num.num  num:num.num'

C:\Users\savel\AppData\Local\Temp\ipykernel_8764\1441048950.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['CS:'] = df['CS:'].apply(hour_norm)
C:\Users\savel\AppData\Local\Temp\ipykernel_8764\1441048950.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['CS:'] = df['CS:'].apply(hour_norm)
C:\Users\savel\AppData\Local\Temp\ipykernel_8764\1441048950.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value i